In [7]:
# CODE CELL 1: Setup and Data Loading
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from google.colab import files

# Download stopwords for text cleaning
nltk.download('stopwords')

df = pd.read_csv("/content/WELFake_Dataset.csv",on_bad_lines='skip', engine='python')

# Display basic info
print("\n--- Dataset Info ---")
print(df.head())
print(f"\nTotal records: {len(df)}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



--- Dataset Info ---
  Unnamed: 0                                              title  \
0          0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1          1                                                NaN   
2          2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3          3  Bobby Jindal, raised Hindu, uses story of Chri...   
4          4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text label  
0  No comment is expected from Barack Obama Membe...     1  
1     Did they post their votes for Hillary already?     1  
2   Now, most of the demonstrators gathered last ...     1  
3  A dozen politically active pastors came here f...     0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...     1  

Total records: 5588


In [9]:
print("Starting Data Preprocessing...")

# Drop redundant columns if they exist
columns_to_drop = ['Unnamed: 0', 'Serial', 'id']
for col in columns_to_drop:
    if col in df.columns:
        df.drop([col], axis=1, inplace=True)

# 1. Handle Missing and Non-Numeric Labels (CRITICAL FIX)
# Convert 'label' to numeric, setting errors='coerce' will turn any corrupted text
# (like the one causing the ValueError) into NaN (Not a Number).
rows_before_drop = len(df)
df['label'] = pd.to_numeric(df['label'], errors='coerce')

# Now drop rows where 'label' is NaN (either originally missing or was corrupted text)
df.dropna(subset=['label'], inplace=True)
rows_after_drop = len(df)
print(f"Dropped {rows_before_drop - rows_after_drop} rows due to missing or corrupted labels.")

# 2. Handle Missing Text Features
# Fill missing 'title' and 'text' with empty strings
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')

# 3. Combine 'title' and 'text' for richer features
df['content'] = df['title'] + ' ' + df['text']

# 4. Text Cleaning Function
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase and ensure it's a string
    text = str(text).lower()
    # Remove special characters/numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove stop words (optional, but good for classical ML)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Apply cleaning
df['content'] = df['content'].apply(clean_text)

# 5. Define features (X) and target (y)
X = df['content']
# Ensure y is an integer type (now safe, as all non-numeric values were converted to NaN and dropped)
y = df['label'].astype(int)

print("Preprocessing complete. Sample cleaned content:")
print(X.iloc[0][:200] + '...') # Show first 200 characters of the first cleaned article


Starting Data Preprocessing...
Dropped 1 rows due to missing or corrupted labels.
Preprocessing complete. Sample cleaned content:
law enforcement high alert following threats cops whites blacklivesmatter fyf terrorists video comment expected barack obama members fyf fukyoflag blacklivesmatter movements called lynching hanging wh...


In [10]:
# Split the data into training and testing sets (80% train, 20% test)
# stratify=y ensures the proportion of real/fake news is the same in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Initialize a TfidfVectorizer
# We use ngram_range=(1, 2) to capture both single words and common two-word phrases
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english', max_df=0.70, ngram_range=(1, 2)
)

# Fit and transform the training data
print("Fitting TF-IDF Vectorizer to training data...")
tfidf_train = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data using the *fitted* vectorizer
print("Transforming test data...")
tfidf_test = tfidf_vectorizer.transform(X_test)

print(f"\nShape of TF-IDF matrix (Train): {tfidf_train.shape} (Features: {tfidf_train.shape[1]})")
print(f"Shape of TF-IDF matrix (Test): {tfidf_test.shape}")


Training samples: 4457
Testing samples: 1115
Fitting TF-IDF Vectorizer to training data...
Transforming test data...

Shape of TF-IDF matrix (Train): (4457, 974966) (Features: 974966)
Shape of TF-IDF matrix (Test): (1115, 974966)


In [11]:
print("Starting Model Training with Passive Aggressive Classifier...")

# Initialize the PassiveAggressiveClassifier
# max_iter=50 sets the number of passes over the training data
pac = PassiveAggressiveClassifier(max_iter=50, random_state=42, n_jobs=-1)

# Train the model
pac.fit(tfidf_train, y_train)

print("Training complete. Making predictions on the test set...")

# Predict on the test set
y_pred = pac.predict(tfidf_test)

# Calculate accuracy
score = accuracy_score(y_test, y_pred)

print(f"\nModel Training & Prediction Complete.")
print(f"Classification Accuracy: {round(score*100, 2)}%")

# Generate Confusion Matrix and Classification Report
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=['Actual Fake (0)', 'Actual Real (1)'],
    columns=['Predicted Fake (0)', 'Predicted Real (1)']
)
print(cm_df)

print("\n--- Classification Report (0: Fake, 1: Real) ---")
print(classification_report(y_test, y_pred, target_names=['Fake (0)', 'Real (1)']))



Starting Model Training with Passive Aggressive Classifier...
Training complete. Making predictions on the test set...

Model Training & Prediction Complete.
Classification Accuracy: 93.27%

--- Confusion Matrix ---
                 Predicted Fake (0)  Predicted Real (1)
Actual Fake (0)                 502                  27
Actual Real (1)                  48                 538

--- Classification Report (0: Fake, 1: Real) ---
              precision    recall  f1-score   support

    Fake (0)       0.91      0.95      0.93       529
    Real (1)       0.95      0.92      0.93       586

    accuracy                           0.93      1115
   macro avg       0.93      0.93      0.93      1115
weighted avg       0.93      0.93      0.93      1115



In [13]:
print("\n--- Testing Model with Custom Examples ---")

# Define the function to run new text through the pipeline
def predict_fake_news(news_text, model, vectorizer):
    # 1. Clean the text using the same function used during training
    cleaned_text = clean_text(news_text)

    # 2. Vectorize the text using the *fitted* TF-IDF Vectorizer
    vectorized_text = vectorizer.transform([cleaned_text])

    # 3. Predict the label
    prediction = model.predict(vectorized_text)[0]

    # 4. Return result
    result = "REAL NEWS (Label 1) ✅" if prediction == 1 else "FAKE NEWS (Label 0) ❌"
    return result

# --- Test Case 1: Example of a potentially real headline/article ---
test_article_1 = "The U.S. Federal Reserve announced today it will hold interest rates steady following a review of current employment data and inflation forecasts."
print(f"\nArticle 1 (Real Candidate): {test_article_1[:70]}...")
print(f"Prediction: {predict_fake_news(test_article_1, pac, tfidf_vectorizer)}")

# --- Test Case 2: Example of a potentially fake headline/article ---
test_article_2 = "Scientists discover that eating only ice cream cures all diseases, and the medical establishment is trying to cover it up."
print(f"\nArticle 2 (Fake Candidate): {test_article_2[:70]}...")
print(f"Prediction: {predict_fake_news(test_article_2, pac, tfidf_vectorizer)}")


--- Testing Model with Custom Examples ---

Article 1 (Real Candidate): The U.S. Federal Reserve announced today it will hold interest rates s...
Prediction: REAL NEWS (Label 1) ✅

Article 2 (Fake Candidate): Scientists discover that eating only ice cream cures all diseases, and...
Prediction: REAL NEWS (Label 1) ✅


In [14]:
import joblib
from google.colab import files

# Save the TfidfVectorizer
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
print("Saved tfidf_vectorizer.pkl")

# Save the PassiveAggressiveClassifier model
joblib.dump(pac, 'pac_model.pkl')
print("Saved pac_model.pkl")

# Download the files to your local machine
print("\nDownloading model artifacts...")
files.download('tfidf_vectorizer.pkl')
files.download('pac_model.pkl')
print("Download complete. Check your browser's download folder.")

Saved tfidf_vectorizer.pkl
Saved pac_model.pkl



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete. Check your browser's download folder.
